# Demo 04 : A tool call by hand

Run the cells in order and watch. There is nothing to write.

Run the next cell first. Its code is folded : click the cell and press Shift+Enter. It loads the course key and builds `llm`, the course model.

In [2]:
#@title Course setup : run this cell first
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · the-request

The model receives the tool descriptions. Its reply contains a request to call a tool. Nothing has run yet.

In [3]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]
SECTIONS

['# Writing a canvas game\n## One file\n\nA game is one HTML file with no outside scripts, so anyone can open it by double-clicking. The file has four script\nblocks : theme, logic, bot and view.\n',
 '# Writing a canvas game\n## Pure logic\n\nAll rules live in the logic block as two functions, `init(seed)` and `step(state, input)`. The logic has no DOM, no clock\nand no `Math.random`. That is what lets the Tester run the game headless through QuickJS, thousands of ticks in a second.\n',
 '# Writing a canvas game\n## State\n\nThe state is a plain JSON object. `step` returns a new state and leaves the old one untouched, so a replay is exact and a\nbuild can be rolled back to any tick.\n',
 '# Writing a canvas game\n## Common mistakes\n\nChanging the state that came in, which breaks replay. Removing the tail twice before the self hit check, which lets the\nhead pass through the last body cell. Replacing the comet list after a spawn, which throws away the comet that just\nspawned. Checkin

In [4]:
def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

tools = [game_info, assets_for_game]
by_name = {t.name: t for t in tools}
tools

[StructuredTool(name='game_info', description='Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch.', args_schema=<class 'langchain_core.utils.pydantic.game_info'>, func=<function game_info at 0x0000019A9D974F40>),
 StructuredTool(name='assets_for_game', description='List sprites, palettes and sounds for a game id such as CS-01, with colour and status.', args_schema=<class 'langchain_core.utils.pydantic.assets_for_game'>, func=<function assets_for_game at 0x0000019A9A52FC40>)]

In [5]:
model = llm.bind_tools(tools)
messages = [("human", "What genre is Comet Snake? Use a tool.")]
reply = model.invoke(messages)

print("text:", repr(reply.content))
print("tool calls:", reply.tool_calls)

text: ''
tool calls: [{'name': 'game_info', 'args': {'title': 'Comet Snake'}, 'id': 'call_8NZDRj4HyB7XTTJi5eKc19F1', 'type': 'tool_call'}]


## b · the-result

Our code runs the function and sends the result back as a tool message. The model then answers from that result.

In [8]:
from langchain.messages import ToolMessage

call = reply.tool_calls[0]
result = by_name[call["name"]].invoke(call["args"])
print("our code ran", call["name"], "->", result)

messages += [reply, ToolMessage(result, tool_call_id=call["id"])]
final = model.invoke(messages)

print("text:", final.content)
print("tool calls:", final.tool_calls)

our code ran game_info -> {"game_id": "CS-01", "title": "Comet Snake", "genre": "arcade", "status": "in development", "pitch": "Snake in space : eat stars, dodge comets, every fifth star speeds up"}
text: Comet Snake’s genre is arcade.
tool calls: []


In [7]:
messages

[('human', 'What genre is Comet Snake? Use a tool.'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 108, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 2.08e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.08e-05, 'upstream_inference_prompt_cost': 1.08e-05, 'upstream_inference_completions_cost': 1e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-6-luna', 'system_fingerprint': None, 'id': 'gen-1793615087-Nk4FsbmOY780TDPWASTA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19bca-4928-7e53-93ce-299a3ae4a8dd-0

## Things to try

Optional : we can change the sentence or the question and run the example again.